# Lesson 14A: Self-Supervised Learning - Theory

**Self-supervised** learns representations without labels.

**Contrastive learning**: Pull similar samples together, push dissimilar apart.

**SimCLR**: Contrastive learning with data augmentation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print('✅ Loaded!')

## NT-Xent Loss

ℓ(i,j) = -log [exp(sim(z_i, z_j)/τ) / Σ_k exp(sim(z_i, z_k)/τ)]

In [ ]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 128))
        self.projection = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 64))
    
    def forward(self, x):
        h = self.encoder(x)
        z = self.projection(h)
        return h, F.normalize(z, dim=-1)

model = SimCLR()
print('✅ SimCLR defined')

In [ ]:
def nt_xent_loss(z_i, z_j, temp=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim = torch.mm(z, z.T) / temp
    mask = torch.eye(2*batch_size, dtype=torch.bool)
    sim = sim.masked_fill(mask, -9e15)
    pos_sim = torch.cat([torch.diag(sim, batch_size), torch.diag(sim, -batch_size)])
    loss = -pos_sim + torch.logsumexp(sim, dim=1)
    return loss.mean()

z_i = torch.randn(32, 64)
z_j = torch.randn(32, 64)
loss = nt_xent_loss(z_i, z_j)
print(f'NT-Xent loss: {loss.item():.3f}')
print('✅ Contrastive loss defined')